<a href="https://colab.research.google.com/github/Brianoyaro/ml_zoomcamp/blob/main/pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/alexeygrigorev/clothing-dataset-small.git

Cloning into 'clothing-dataset-small'...
remote: Enumerating objects: 3839, done.
remote: Counting objects: 100% (400/400), done.
remote: Compressing objects: 100% (400/400), done.
remote: Total 3839 (delta 9), reused 385 (delta 0), pack-reused 3439 (from 1)
Receiving objects: 100% (3839/3839), 100.58 MiB | 14.34 MiB/s, done.
Resolving deltas: 100% (10/10), done.
Updating files: 100% (3783/3783), done.


In [2]:
import os
from torch.utils.data import Dataset
from PIL import Image

# A python dataset for image loading
class ClothingDataset(Dataset):
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        self.image_paths = []
        self.labels = []
        self.classes = sorted(os.listdir(data_dir))
        self.class_to_idx = {cls: i for i, cls in enumerate(self.classes)}

        for label_name in self.classes:
            label_dir = os.path.join(data_dir, label_name)
            for img_name in os.listdir(label_dir):
                self.image_paths.append(os.path.join(label_dir, img_name))
                self.labels.append(self.class_to_idx[label_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label

In [3]:
# Simple pre-processing

from torchvision import transforms

input_size = 224

# ImageNet normalization values
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]

# Simple transforms - just resize and normalize
train_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

val_transforms = transforms.Compose([
    transforms.Resize((input_size, input_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

In [4]:
# Data Loader

from torch.utils.data import DataLoader

train_dataset = ClothingDataset(
    data_dir='./clothing-dataset-small/train',
    transform=train_transforms
)

val_dataset = ClothingDataset(
    data_dir='./clothing-dataset-small/validation',
    transform=val_transforms
)

print(f"train dataset object: {train_dataset}")
print(f"val dataset object: {val_dataset}")

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

train dataset object: <__main__.ClothingDataset object at 0x7cea1d491940>
val dataset object: <__main__.ClothingDataset object at 0x7cea1d64b9d0>


In [5]:
# Build the model

import torch.nn as nn
import torchvision.models as models

class ClothingClassifierMobileNet(nn.Module):
    def __init__(self, num_classes=10):
        super(ClothingClassifierMobileNet, self).__init__()

        # Load pre-trained MobileNetV2
        self.base_model = models.mobilenet_v2(weights='IMAGENET1K_V1')

        # Freeze base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Remove original classifier
        self.base_model.classifier = nn.Identity()

        # Add custom layers
        self.global_avg_pooling = nn.AdaptiveAvgPool2d((1, 1))
        self.output_layer = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.base_model.features(x)
        x = self.global_avg_pooling(x)
        x = torch.flatten(x, 1)
        x = self.output_layer(x)
        return x

In [6]:
import torch
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ClothingClassifierMobileNet(num_classes=10)
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 95.0MB/s]


In [7]:
'''# Training loop
num_epochs = 10

for epoch in range(num_epochs):
    # Training phase
    model.train()  # Set the model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Iterate over the training data
    for inputs, labels in train_loader:
        # Move data to the specified device (GPU or CPU)
        inputs, labels = inputs.to(device), labels.to(device)

        # Zero the parameter gradients to prevent accumulation
        optimizer.zero_grad()
        # Forward pass
        outputs = model(inputs)
        # Calculate the loss
        loss = criterion(outputs, labels)
        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        # Accumulate training loss
        running_loss += loss.item()
        # Get predictions
        _, predicted = torch.max(outputs.data, 1)
        # Update total and correct predictions
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # Calculate average training loss and accuracy
    train_loss = running_loss / len(train_loader)
    train_acc = correct / total

    # Validation phase
    model.eval()  # Set the model to evaluation mode
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    # Disable gradient calculation for validation
    with torch.no_grad():
        # Iterate over the validation data
        for inputs, labels in val_loader:
            # Move data to the specified device (GPU or CPU)
            inputs, labels = inputs.to(device), labels.to(device)
            # Forward pass
            outputs = model(inputs)
            # Calculate the loss
            loss = criterion(outputs, labels)

            # Accumulate validation loss
            val_loss += loss.item()
            # Get predictions
            _, predicted = torch.max(outputs.data, 1)
            # Update total and correct predictions
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    # Calculate average validation loss and accuracy
    val_loss /= len(val_loader)
    val_acc = val_correct / val_total

    # Print epoch results
    print(f'Epoch {epoch+1}/{num_epochs}')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')'''

"# Training loop\nnum_epochs = 10\n\nfor epoch in range(num_epochs):\n    # Training phase\n    model.train()  # Set the model to training mode\n    running_loss = 0.0\n    correct = 0\n    total = 0\n\n    # Iterate over the training data\n    for inputs, labels in train_loader:\n        # Move data to the specified device (GPU or CPU)\n        inputs, labels = inputs.to(device), labels.to(device)\n\n        # Zero the parameter gradients to prevent accumulation\n        optimizer.zero_grad()\n        # Forward pass\n        outputs = model(inputs)\n        # Calculate the loss\n        loss = criterion(outputs, labels)\n        # Backward pass and optimize\n        loss.backward()\n        optimizer.step()\n\n        # Accumulate training loss\n        running_loss += loss.item()\n        # Get predictions\n        _, predicted = torch.max(outputs.data, 1)\n        # Update total and correct predictions\n        total += labels.size(0)\n        correct += (predicted == labels).sum().

In [8]:
def train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device):
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total

        print(f'Epoch {epoch+1}/{num_epochs}')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
        print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

In [9]:
train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs=10, device=device)

Epoch 1/10
  Train Loss: 1.4178, Train Acc: 0.6362
  Val Loss: 1.0997, Val Acc: 0.6657
Epoch 2/10
  Train Loss: 0.7530, Train Acc: 0.7728
  Val Loss: 0.9183, Val Acc: 0.7273
Epoch 3/10
  Train Loss: 0.6666, Train Acc: 0.7953
  Val Loss: 1.3712, Val Acc: 0.6891
Epoch 4/10
  Train Loss: 0.6748, Train Acc: 0.8077
  Val Loss: 0.8097, Val Acc: 0.7830
Epoch 5/10
  Train Loss: 0.5178, Train Acc: 0.8422
  Val Loss: 1.2913, Val Acc: 0.7009
Epoch 6/10
  Train Loss: 0.4853, Train Acc: 0.8527
  Val Loss: 1.1928, Val Acc: 0.7302
Epoch 7/10
  Train Loss: 0.5763, Train Acc: 0.8455
  Val Loss: 0.9149, Val Acc: 0.7859
Epoch 8/10
  Train Loss: 0.4803, Train Acc: 0.8579
  Val Loss: 1.3958, Val Acc: 0.7214
Epoch 9/10
  Train Loss: 0.4944, Train Acc: 0.8592
  Val Loss: 0.9903, Val Acc: 0.7537
Epoch 10/10
  Train Loss: 0.6978, Train Acc: 0.8286
  Val Loss: 1.0949, Val Acc: 0.7683


In [11]:
# Tunig the learning rate

def make_model(learning_rate=0.01):
    model = ClothingClassifierMobileNet(num_classes=10)
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    return model, optimizer

learning_rates = [0.0001, 0.001, 0.01, 0.1]
num_epochs = 10

for lr in learning_rates:
    print(f'\n=== Learning Rate: {lr} ===')
    model, optimizer = make_model(learning_rate=lr)
    train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device)


=== Learning Rate: 0.0001 ===
Epoch 1/10
  Train Loss: 1.9850, Train Acc: 0.3286
  Val Loss: 1.8008, Val Acc: 0.4399
Epoch 2/10
  Train Loss: 1.6288, Train Acc: 0.5042
  Val Loss: 1.5154, Val Acc: 0.5220
Epoch 3/10
  Train Loss: 1.4005, Train Acc: 0.5838
  Val Loss: 1.3267, Val Acc: 0.5836
Epoch 4/10
  Train Loss: 1.2434, Train Acc: 0.6379
  Val Loss: 1.1911, Val Acc: 0.6452
Epoch 5/10
  Train Loss: 1.1219, Train Acc: 0.6835
  Val Loss: 1.0848, Val Acc: 0.6686
Epoch 6/10
  Train Loss: 1.0357, Train Acc: 0.7177
  Val Loss: 1.0129, Val Acc: 0.6979
Epoch 7/10
  Train Loss: 0.9672, Train Acc: 0.7373
  Val Loss: 0.9503, Val Acc: 0.6979
Epoch 8/10
  Train Loss: 0.9075, Train Acc: 0.7493
  Val Loss: 0.8978, Val Acc: 0.7214
Epoch 9/10
  Train Loss: 0.8523, Train Acc: 0.7744
  Val Loss: 0.8592, Val Acc: 0.7390
Epoch 10/10
  Train Loss: 0.8198, Train Acc: 0.7777
  Val Loss: 0.8327, Val Acc: 0.7243

=== Learning Rate: 0.001 ===
Epoch 1/10
  Train Loss: 1.2781, Train Acc: 0.5975
  Val Loss: 0.801

In [12]:
best_learning_rate = 0.001

In [13]:
# model checkpointing - saving the best model while training

def train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device):
    best_val_accuracy = 0.0  # Initialize variable to track the best validation accuracy

    for epoch in range(num_epochs):
        # Training phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        train_loss = running_loss / len(train_loader)
        train_acc = correct / total

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)

                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, predicted = torch.max(outputs.data, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        val_loss /= len(val_loader)
        val_acc = val_correct / val_total

        print(f'Epoch {epoch+1}/{num_epochs}')
        print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
        print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')

        # Checkpoint the model if validation accuracy improved
        if val_acc > best_val_accuracy:
            best_val_accuracy = val_acc
            checkpoint_path = f'mobilenet_v2_{epoch+1:02d}_{val_acc:.3f}.pth'
            torch.save(model.state_dict(), checkpoint_path)
            print(f'Checkpoint saved: {checkpoint_path}')

In [14]:
# training model with checkpoint enabled

model, optimizer = make_model(learning_rate=best_learning_rate)
train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device)

Epoch 1/10
  Train Loss: 1.3311, Train Acc: 0.5730
  Val Loss: 0.8628, Val Acc: 0.7214
Checkpoint saved: mobilenet_v2_01_0.721.pth
Epoch 2/10
  Train Loss: 0.7712, Train Acc: 0.7595
  Val Loss: 0.6683, Val Acc: 0.7625
Checkpoint saved: mobilenet_v2_02_0.762.pth
Epoch 3/10
  Train Loss: 0.6197, Train Acc: 0.8038
  Val Loss: 0.6033, Val Acc: 0.7889
Checkpoint saved: mobilenet_v2_03_0.789.pth
Epoch 4/10
  Train Loss: 0.5692, Train Acc: 0.8207
  Val Loss: 0.6399, Val Acc: 0.8006
Checkpoint saved: mobilenet_v2_04_0.801.pth
Epoch 5/10
  Train Loss: 0.5100, Train Acc: 0.8409
  Val Loss: 0.5778, Val Acc: 0.8123
Checkpoint saved: mobilenet_v2_05_0.812.pth
Epoch 6/10
  Train Loss: 0.4665, Train Acc: 0.8527
  Val Loss: 0.6232, Val Acc: 0.7771
Epoch 7/10
  Train Loss: 0.4581, Train Acc: 0.8589
  Val Loss: 0.6112, Val Acc: 0.7654
Epoch 8/10
  Train Loss: 0.4125, Train Acc: 0.8696
  Val Loss: 0.5441, Val Acc: 0.8035
Epoch 9/10
  Train Loss: 0.3997, Train Acc: 0.8677
  Val Loss: 0.5651, Val Acc: 0.80

In [17]:
# Adding inner layers

import torch.nn as nn
import torchvision.models as models

class ClothingClassifierMobileNet(nn.Module):
    def __init__(self, size_inner=100, num_classes=10):
        super(ClothingClassifierMobileNet, self).__init__()

        # Load pre-trained MobileNetV2
        self.base_model = models.mobilenet_v2(weights='IMAGENET1K_V1')

        # Freeze base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Remove original classifier
        self.base_model.classifier = nn.Identity()

        # Add custom layers
        self.global_avg_pooling = nn.AdaptiveAvgPool2d((1, 1))

        self.inner = nn.Linear(1280, size_inner)  # New inner layer
        self.relu = nn.ReLU()

        self.output_layer = nn.Linear(size_inner, num_classes)

    def forward(self, x):
        x = self.base_model.features(x)
        x = self.global_avg_pooling(x)
        x = torch.flatten(x, 1)

        x = self.inner(x)
        x = self.relu(x)

        x = self.output_layer(x)
        return x



def make_model(learning_rate=0.001, size_inner=100):
    model = ClothingClassifierMobileNet(
        num_classes=10,
        size_inner=size_inner
    )
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    return model, optimizer


model, optimizer = make_model(learning_rate=best_learning_rate)
train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device)

Epoch 1/10
  Train Loss: 1.2564, Train Acc: 0.5971
  Val Loss: 0.7945, Val Acc: 0.7302
Checkpoint saved: mobilenet_v2_01_0.730.pth
Epoch 2/10
  Train Loss: 0.6886, Train Acc: 0.7764
  Val Loss: 0.6418, Val Acc: 0.7977
Checkpoint saved: mobilenet_v2_02_0.798.pth
Epoch 3/10
  Train Loss: 0.5623, Train Acc: 0.8103
  Val Loss: 0.5992, Val Acc: 0.7771
Epoch 4/10
  Train Loss: 0.5260, Train Acc: 0.8227
  Val Loss: 0.5830, Val Acc: 0.7889
Epoch 5/10
  Train Loss: 0.4590, Train Acc: 0.8468
  Val Loss: 0.5692, Val Acc: 0.7977
Epoch 6/10
  Train Loss: 0.4347, Train Acc: 0.8452
  Val Loss: 0.5300, Val Acc: 0.8065
Checkpoint saved: mobilenet_v2_06_0.806.pth
Epoch 7/10
  Train Loss: 0.3981, Train Acc: 0.8598
  Val Loss: 0.5453, Val Acc: 0.7977
Epoch 8/10
  Train Loss: 0.3757, Train Acc: 0.8729
  Val Loss: 0.5877, Val Acc: 0.7801
Epoch 9/10
  Train Loss: 0.3499, Train Acc: 0.8778
  Val Loss: 0.5842, Val Acc: 0.7859
Epoch 10/10
  Train Loss: 0.3205, Train Acc: 0.8869
  Val Loss: 0.5853, Val Acc: 0.79

In [ ]:
# Drop-out and regularization


import torch.nn as nn
import torchvision.models as models

class ClothingClassifierMobileNet(nn.Module):
    def __init__(self, size_inner=100, num_classes=10):
        super(ClothingClassifierMobileNet, self).__init__()

        # Load pre-trained MobileNetV2
        self.base_model = models.mobilenet_v2(weights='IMAGENET1K_V1')

        # Freeze base model parameters
        for param in self.base_model.parameters():
            param.requires_grad = False

        # Remove original classifier
        self.base_model.classifier = nn.Identity()

        # Add custom layers
        self.global_avg_pooling = nn.AdaptiveAvgPool2d((1, 1))

        self.inner = nn.Linear(1280, size_inner)  # New inner layer
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(droprate)  # Add dropout

        self.output_layer = nn.Linear(size_inner, num_classes)

    def forward(self, x):
        x = self.base_model.features(x)
        x = self.global_avg_pooling(x)
        x = torch.flatten(x, 1)

        x = self.inner(x)
        x = self.relu(x)
        x = self.dropout(x)  # Apply dropout

        x = self.output_layer(x)
        return x



def make_model(
        learning_rate=0.001,
        size_inner=100,
        droprate=0.2
):
    model = ClothingClassifierMobileNet(
        num_classes=10,
        size_inner=size_inner,
        droprate=droprate
    )
    model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    return model, optimizer


model, optimizer = make_model(learning_rate=best_learning_rate)
train_and_evaluate(model, optimizer, train_loader, val_loader, criterion, num_epochs, device)

In [20]:
# Data Augmentation

# Training transforms WITH augmentation
train_transforms_aug = transforms.Compose([
    transforms.RandomRotation(10),           # Rotate up to 10 degrees
    transforms.RandomResizedCrop(224, scale=(0.9, 1.0)),  # Zoom
    transforms.RandomHorizontalFlip(),       # Horizontal flip
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])

# Validation transforms - NO augmentation, same as before
val_transforms_aug = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=mean, std=std)
])


# Data Loader

from torch.utils.data import DataLoader

train_dataset_aug = ClothingDataset(
    data_dir='./clothing-dataset-small/train',
    transform=train_transforms_aug
)

val_dataset_aug = ClothingDataset(
    data_dir='./clothing-dataset-small/validation',
    transform=val_transforms_aug
)

print(f"train dataset  Augmented object: {train_dataset_aug}")
print(f"val dataset Augmented object: {val_dataset_aug}")

train_loader_aug = DataLoader(train_dataset_aug, batch_size=32, shuffle=True)
val_loader_aug = DataLoader(val_dataset_aug, batch_size=32, shuffle=False)


model, optimizer = make_model(learning_rate=best_learning_rate)
train_and_evaluate(model, optimizer, train_loader_aug, val_loader_aug, criterion, num_epochs, device)

train dataset  Augmented object: <__main__.ClothingDataset object at 0x7cea09880d60>
val dataset Augmented object: <__main__.ClothingDataset object at 0x7cea1d4cc950>
Epoch 1/10
  Train Loss: 1.3007, Train Acc: 0.5779
  Val Loss: 0.7744, Val Acc: 0.7478
Checkpoint saved: mobilenet_v2_01_0.748.pth
Epoch 2/10
  Train Loss: 0.7598, Train Acc: 0.7493
  Val Loss: 0.7443, Val Acc: 0.7507
Checkpoint saved: mobilenet_v2_02_0.751.pth
Epoch 3/10
  Train Loss: 0.6510, Train Acc: 0.7771
  Val Loss: 0.6229, Val Acc: 0.7889
Checkpoint saved: mobilenet_v2_03_0.789.pth
Epoch 4/10
  Train Loss: 0.5746, Train Acc: 0.8057
  Val Loss: 0.5631, Val Acc: 0.8035
Checkpoint saved: mobilenet_v2_04_0.804.pth
Epoch 5/10
  Train Loss: 0.5533, Train Acc: 0.8067
  Val Loss: 0.5847, Val Acc: 0.7859
Epoch 6/10
  Train Loss: 0.5266, Train Acc: 0.8188
  Val Loss: 0.5843, Val Acc: 0.8035
Epoch 7/10
  Train Loss: 0.4831, Train Acc: 0.8393
  Val Loss: 0.6241, Val Acc: 0.7713
Epoch 8/10
  Train Loss: 0.4735, Train Acc: 0.82

In [21]:
# Using trained model

# 1.) Loading saved model
# --------------------------------------------------------------------------------------------------------------------------
import glob
# Find best checkpoint
list_of_files = glob.glob('mobilenet_v2_*.pth')
latest_file = max(list_of_files, key=os.path.getctime)
print(f"Loading model from: {latest_file}")
# Load model
model = ClothingClassifierMobileNet(size_inner=32, droprate=0.2, num_classes=10)
model.load_state_dict(torch.load(latest_file))
model.to(device)
model.eval()



# 2.) Making predictions
# ------------------------------------------------------------------------------------------------------------------------------
from keras_image_helper import create_preprocessor
import numpy as np
def preprocess_pytorch_style(X):
    # X: shape (1, 224, 224, 3), dtype=float32, values in [0, 255]
    X = X / 255.0

    mean = np.array([0.485, 0.456, 0.406]).reshape(1, 3, 1, 1)
    std = np.array([0.229, 0.224, 0.225]).reshape(1, 3, 1, 1)

    # Convert NHWC → NCHW (batch, height, width, channels → batch, channels, height, width)
    X = X.transpose(0, 3, 1, 2)

    # Normalize
    X = (X - mean) / std

    return X.astype(np.float32)

preprocessor = create_preprocessor(preprocess_pytorch_style, target_size=(224, 224))

# Predict from URL
url = 'http://bit.ly/mlbookcamp-pants'
X = preprocessor.from_url(url)
X = torch.Tensor(X).to(device)

with torch.no_grad():
    pred = model(X).cpu().numpy()[0]

classes = [
    "dress", "hat", "longsleeve", "outwear", "pants",
    "shirt", "shoes", "shorts", "skirt", "t-shirt"
]

result = dict(zip(classes, pred.tolist()))
print(result)

Loading model from: mobilenet_v2_09_0.812.pth


TypeError: ClothingClassifierMobileNet.__init__() got an unexpected keyword argument 'droprate'

In [ ]:
# Exporting model to ONNX

# Create dummy input
dummy_input = torch.randn(1, 3, 224, 224).to(device)

# Export to ONNX
onnx_path = "clothing_classifier_mobilenet_v2.onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_path,
    verbose=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"Model exported to {onnx_path}")